# 🌐 AI Website Summarizer — Open-Source Llama 3.2 (1B)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mitadrudeb/ai-website-summarizer/blob/main/Summarizing_websites_using_AI.ipynb)

**Author:** Mitadru Deb  
**Focus:** LLM Application Engineering & Open-Source AI  
**Model:** [Llama 3.2 1B Instruct](https://huggingface.co/unsloth/Llama-3.2-1B-Instruct) (~1.2B Parameters)  
**Last Updated:** November 2026

---

## 📌 Project Overview

This notebook builds an **end-to-end AI website summarizer** using **Meta's Llama 3.2 (1B Instruct)**.

- **Input:** Any webpage URL (Wikipedia, documentation, blogs, news articles)
- **Pipeline:** HTML scraping & boilerplate removal → prompt construction → local open-source LLM inference
- **Output:** Concise, structured, plain-English summary with key announcements
- **Zero API Cost:** 100% open-source, runs on CPU or GPU without paid OpenAI API keys

---

## 1. Setup & Installation

Install the core libraries for web scraping, deep learning, and transformer inference.

In [ ]:
# Install required dependencies
!pip install -q transformers torch torchvision requests beautifulsoup4 accelerate

## 2. Import Libraries & Check Environment

We inspect the hardware to automatically route inference to CUDA GPU (if available) or standard CPU.

In [ ]:
import os
import platform
import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display

import torch
from transformers import pipeline

# Verify hardware device
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"OS Platform:     {platform.system()} {platform.release()}")
print(f"PyTorch Version: {torch.__version__}")
print(f"Target Device:   {device.upper()}")
print(f"Tensor Dtype:    {torch_dtype}")
if torch.cuda.is_available():
    print(f"GPU Name:        {torch.cuda.get_device_name(0)}")

## 3. Web Scraping & HTML Cleaning

Before feeding website text into an LLM, we must strip out non-content elements like scripts, styles, navigation bars, and footer clutter.

In [ ]:
# Standard user-agent header to avoid basic scraping blocks
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

def fetch_website_contents(url: str, max_chars: int = 2500) -> str:
    """
    Fetch the webpage, strip out HTML boilerplate, and return clean readable text.
    """
    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()
    except requests.RequestException as e:
        return f"Error fetching webpage: {e}"

    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string.strip() if soup.title and soup.title.string else "No title found"

    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input", "noscript", "svg", "footer", "header"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""

    return f"Title: {title}\n\nContent:\n{text}"[:max_chars]


# Quick test on a Wikipedia article
test_sample = fetch_website_contents("https://en.wikipedia.org/wiki/James_Bond")
print(f"Extracted {len(test_sample)} characters:\n")
print(test_sample[:400] + "...")

## 4. Load Open-Source Llama 3.2 (1B Instruct)

We load **Llama 3.2 1B Instruct** using the Hugging Face `text-generation` pipeline.

> 💡 **Why unsloth/Llama-3.2-1B-Instruct?**  
> It is an ungated, direct mirror of Meta's official weights. It allows zero-friction downloading without needing to wait for a gated license approval.

In [ ]:
model_id = "unsloth/Llama-3.2-1B-Instruct"
print(f"Loading {model_id} (first run downloads weights)...\n")

summarizer_pipe = pipeline(
    "text-generation",
    model=model_id,
    torch_dtype=torch_dtype,
    device=device,
)

print("✅ Llama 3.2 1B successfully loaded into memory!")

## 5. Prompt Engineering for Summarization

Llama 3.2 is an instruction-tuned conversational model. We define:
1. **System Prompt:** Instructs the model to act as a skilled English professor who explains complex content in everyday, plain English.
2. **User Prompt:** Provides the scraped text and asks for a structured summary + announcements.

In [ ]:
system_prompt = """
You are an experienced English professor who excels in summarization of any English language content.
You will be given the contents of a website and your job is to summarize it using simple, everyday English
in such a way that it is easily understood by anyone. If the page includes announcements, news, or updates, highlight them.
Format your response in clean Markdown.
"""

user_prompt = """
Here are the contents of a website. Please provide a clear, structured summary:

"""

def messages_for(website_content: str):
    return [
        {"role": "system", "content": system_prompt.strip()},
        {"role": "user", "content": user_prompt + website_content},
    ]

print("✅ Prompt templates ready.")

## 6. End-to-End Summarization Pipeline

This function ties together the entire workflow: **URL Ingestion → HTML Cleaning → Prompting → Llama 3.2 Generation → Markdown Display**.

In [ ]:
def summarize(url: str, max_new_tokens: int = 300) -> str:
    """
    Fetch webpage contents and generate an open-source summary using Llama 3.2.
    """
    website_content = fetch_website_contents(url)
    if not website_content or not website_content.strip():
        return "⚠️ Could not extract text from this website."

    messages = messages_for(website_content)
    outputs = summarizer_pipe(
        messages,
        max_new_tokens=max_new_tokens,
        do_sample=False,  # Deterministic decoding for consistent summaries
    )
    summary = outputs[0]["generated_text"][-1]["content"]
    return summary


def display_summary(url: str):
    """
    Helper to render formatted Markdown directly in the Jupyter notebook.
    """
    print(f"⏳ Summarizing: {url}...")
    result = summarize(url)
    display(Markdown(f"### 📝 Summary for: {url}\n\n" + result))

print("✅ summarize() and display_summary() defined.")

## 7. Real-World Demonstrations

Let's test our open-source summarizer on diverse real-world websites.

In [ ]:
# Test 1: Simple practice website
display_summary("https://www.scrapethissite.com/pages/simple/")

In [ ]:
# Test 2: Content-rich encyclopedia page
display_summary("https://en.wikipedia.org/wiki/Python_(programming_language)")

## 8. What I Learned & Engineering Takeaways

### 💡 Key Takeaways for LLM Engineers

1. **Open-Source vs Proprietary APIs:** Migrating from paid closed-source APIs (like OpenAI GPT-4) to open-source models (like Meta Llama 3.2 1B) eliminates per-token costs, prevents vendor lock-in, and guarantees total data privacy.

2. **1B Parameter Efficiency:** At ~1.2B parameters, Llama 3.2 is compact enough to run locally on consumer hardware while still retaining impressive reasoning, instruction following, and language summarization capabilities.

3. **Chat Templates:** Modern decoder-only LLMs require structured conversation templates (`system` + `user` + `assistant`). Passing raw strings without chat templates degrades instruction adherence.

4. **Defensive Web Scraping:** Scraping real websites requires timeouts, user-agent headers, and tag decomposition to avoid overloading context windows with noisy HTML boilerplate.